# Preprocessing and Feature Extraction (chest + wrist)

We build the combined feature table used for stress classification on WESAD, using both wearable
devices: the chest RespiBAN (ECG, EDA, EMG, respiration, temperature, at 700 Hz) and the wrist
Empatica E4 (BVP at 64 Hz, EDA and temperature at 4 Hz). The accelerometer is excluded: it captures
motion, which is confounded with the study protocol (subjects stand and speak during the stress
condition), so a classifier could learn movement patterns rather than the affective state.

Processing follows the order used in Schmidt et al. (2018): each signal is filtered as a whole,
then segmented into overlapping 60-second windows (5-second stride) that stay inside a single
condition, then summarised into features. Windows are defined on the 700 Hz label timeline and
mapped to each signal's own sampling rate, so all signals are aligned within a window. The HRV
metrics follow the WESAD feature set (Schmidt et al., 2018): mean and standard deviation of heart
rate and of the NN (RR) intervals, pNN50, the root mean square of successive NN differences (their
`rmsHRV`, i.e. RMSSD) and the LF/HF ratio, plus the non-linear Poincare and sample-entropy
descriptors from our proposal. The clean chest ECG is used as detected. Only the noisier wrist BVP
is screened for artefacts (Task Force, 1996): RR intervals outside 300-2000 ms (30-200 bpm) or
deviating more than 25% from the local median are flagged, successive-difference metrics use only
differences between originally-adjacent valid intervals so flagging a bad beat never manufactures a
spurious jump, and a window whose valid beats fall below 60% is set to missing rather than reporting
an implausible value.

Every feature is named with a device tag (`c_` for chest, `w_` for wrist) and is additionally
expressed relative to the subject's own baseline (`_rel`), following Setz et al. (2010); baseline
is a known condition, so this per-subject normalisation is legitimate. The wrist EDA is sampled at
4 Hz and cannot resolve the fast phasic peaks, so only its level statistics are used.

The chest feature set (`c_` columns, stripped of the tag) is deliberately kept identical to
`01_preprocessing_features.ipynb` (same signals, same formulas, same feature names), so the
chest-only and combined tables stay comparable.


## Imports

In [1]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd
import neurokit2 as nk
from scipy.signal import welch
from scipy.integrate import trapezoid

warnings.filterwarnings("ignore")

## Configuration

In [ ]:
DATA_DIR = os.path.join(os.getcwd(), "..", "..", "WESAD")   # WESAD folder at the repository root
OUT_DIR = "combined_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

SUBJECTS = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]
# WESAD's raw `label` field spans 0-7 (0 = undefined/transient, 5-7 = unused conditions);
# only the three studied conditions are kept
LABEL_MAP = {1: "baseline", 2: "stress", 3: "amusement"}
KEEP_LABELS = set(LABEL_MAP)

FS_CHEST = 700                       # chest signals
FS_BVP, FS_EDA_W, FS_TEMP_W = 64, 4, 4   # wrist signals
WINDOW_SEC, STRIDE_SEC = 60, 5
# Window/stride length on the 700 Hz chest/label timeline; wrist signals are windowed at their
# own sampling rate (see below), then aligned to the same time span
WLEN, SLEN = WINDOW_SEC * FS_CHEST, STRIDE_SEC * FS_CHEST

TEMP_MIN, TEMP_MAX = 20.0, 45.0      # plausible skin-temperature range (chest)

# Plausible range for breathing rate (breaths/min): below this is apnea-like and above it is
# extreme tachypnea, so values outside are treated as a respiration-detection artefact rather
# than a real physiological rate (see the RESP plausibility check below)
RESP_RATE_MIN, RESP_RATE_MAX = 4.0, 40.0

## Helper functions

Shared building blocks: condition segmentation, non-linear HRV descriptors, band power, and a
statistics block reused for several signals.


In [ ]:
def find_segments(labels):
    """Return (label, start, end) for each contiguous run of a kept condition."""
    segments, i, n = [], 0, len(labels)
    while i < n:
        if labels[i] in KEEP_LABELS:
            j = i
            while j < n and labels[j] == labels[i]:
                j += 1
            segments.append((int(labels[i]), i, j))
            i = j
        else:
            i += 1
    return segments


def sample_entropy(series, m=2, r_factor=0.2):
    """Sample entropy of a short series (e.g. RR intervals)."""
    x = np.asarray(series, float); n = len(x)                   # tolerance, scaled to the series' own variability
    if n < m + 2:
        return np.nan
    r = r_factor * np.std(x)
    if r == 0:
        return np.nan
    def count(dim):
        t = np.array([x[i:i + dim] for i in range(n - dim + 1)]); tot = 0
        for i in range(len(t)):
            tot += np.count_nonzero(np.max(np.abs(t - t[i]), axis=1) <= r) - 1
        return tot
    b, a = count(m), count(m + 1)
    return -np.log(a / b) if a > 0 and b > 0 else np.nan


def poincare_sd(rr):
    """Poincare plot descriptors SD1, SD2 from RR intervals (ms)."""
    if len(rr) < 3:
        return np.nan, np.nan
    sdsd, sdnn = np.std(np.diff(rr), ddof=1), np.std(rr, ddof=1)
    var2 = 2 * sdnn ** 2 - 0.5 * sdsd ** 2
    return np.sqrt(0.5) * sdsd, (np.sqrt(var2) if var2 > 0 else np.nan)


def band_power(signal, fs, low, high):
    """Total power and peak frequency of `signal` within [low, high] Hz (Welch)."""
    nperseg = min(len(signal), fs * 30)
    freqs, psd = welch(signal, fs=fs, nperseg=nperseg)
    band = (freqs >= low) & (freqs <= high)
    if not np.any(band):
        return np.nan, np.nan
    return trapezoid(psd[band], freqs[band]), freqs[band][np.argmax(psd[band])]


HRV_KEYS = ("HR_mean", "HR_std", "HRV_MeanNN", "HRV_SDNN", "HRV_RMSSD", "HRV_pNN50",
            "HRV_SD1", "HRV_SD2", "HRV_SampEn", "HRV_LFHF")

def hrv_features(peak_flags, fs, screen=False, min_valid_frac=0.6):
    """HRV descriptors from beat locations. The metric definitions follow the WESAD feature set
    (Schmidt et al., 2018, Table 1): mean/STD heart rate, mean and STD of the NN (RR) intervals,
    pNN50, the root mean square of successive NN differences (rmsHRV / RMSSD) and the LF/HF power
    ratio. The Poincare SD1/SD2 and sample entropy are the non-linear descriptors added in our
    proposal.

    `screen` toggles RR-interval artefact handling (Task Force, 1996). The clean chest ECG is left
    unedited (`screen=False`), so its values equal a plain RR computation on the detected beats.
    The noisy wrist BVP is screened (`screen=True`): RR intervals outside 300-2000 ms or deviating
    more than 25% from the local median are flagged; successive-difference metrics then use only
    differences between originally-adjacent valid intervals, so removing an artefact never creates a
    spurious jump; and if fewer than 60% of the beats are valid the window is set to NaN rather than
    reporting an implausible value."""
    nan = {k: np.nan for k in HRV_KEYS}
    peaks = np.flatnonzero(np.asarray(peak_flags) == 1)
    if len(peaks) < 4:
        return dict(nan)
    rr = np.diff(peaks) / fs * 1000.0

    if screen:
        valid = (rr >= 300) & (rr <= 2000)
        if valid.sum() >= 3:
            med = np.median(rr[valid])
            valid = valid & (np.abs(rr - med) <= 0.25 * med)
        if valid.mean() < min_valid_frac or valid.sum() < 3:
            return dict(nan)
    else:
        valid = np.ones(len(rr), dtype=bool)                # clean chest ECG: use every interval

    f = {}
    rr_ok = rr[valid]
    hr = 60000.0 / rr_ok
    f["HR_mean"], f["HR_std"] = np.mean(hr), np.std(hr)
    f["HRV_MeanNN"] = np.mean(rr_ok)
    f["HRV_SDNN"] = np.std(rr_ok, ddof=1)

    drr = np.diff(rr)[valid[:-1] & valid[1:]]           # differences of adjacent valid beats only
    if len(drr) >= 1:
        sdsd = np.std(drr, ddof=1) if len(drr) >= 2 else 0.0
        f["HRV_RMSSD"] = np.sqrt(np.mean(drr ** 2))
        f["HRV_pNN50"] = np.mean(np.abs(drr) > 50) * 100.0
        f["HRV_SD1"] = np.sqrt(0.5) * sdsd
        var2 = 2 * f["HRV_SDNN"] ** 2 - 0.5 * sdsd ** 2
        f["HRV_SD2"] = np.sqrt(var2) if var2 > 0 else np.nan
    else:
        f["HRV_RMSSD"] = f["HRV_pNN50"] = f["HRV_SD1"] = f["HRV_SD2"] = np.nan

    f["HRV_SampEn"] = sample_entropy(rr_ok) if len(rr_ok) >= 10 else np.nan
    try:                                                # LF/HF from clean peak times rebuilt from RR
        ps = np.round(np.cumsum(np.concatenate([[0.0], rr_ok])) / 1000.0 * fs).astype(int)
        f["HRV_LFHF"] = float(nk.hrv_frequency({"ECG_R_Peaks": ps}, sampling_rate=fs,
                                              show=False)["HRV_LFHF"].iloc[0])
    except Exception:
        f["HRV_LFHF"] = np.nan
    return f


def temp_features(x):
    """Mean/std/min/max/range/slope of a temperature (or temperature-like) signal."""
    x = np.asarray(x, float)
    return {"TEMP_mean": np.mean(x), "TEMP_std": np.std(x), "TEMP_min": np.min(x),
            "TEMP_max": np.max(x), "TEMP_range": np.max(x) - np.min(x),
            "TEMP_slope": np.polyfit(np.arange(len(x)), x, 1)[0] if len(x) > 1 else 0.0}


def clean_temperature(temp):
    """Replace out-of-range temperature samples (sensor dropout) by interpolation."""
    t = temp.copy()
    t[(t < TEMP_MIN) | (t > TEMP_MAX)] = np.nan
    t = pd.Series(t).interpolate(limit_direction="both").to_numpy()
    return np.nan_to_num(t, nan=np.nanmedian(temp))


def tag(prefix, feats):
    """Prefix every feature name with its device tag."""
    return {f"{prefix}_{k}": v for k, v in feats.items()}

## Per-subject feature extraction

Each subject's chest signals (ECG, EDA, EMG, RESP) are filtered once with `neurokit2`; the wrist
BVP and EDA are processed the same way at their own rates. Then every 60 s window is summarised
into chest (`c_`) and wrist (`w_`) features.


In [ ]:
def process_subject(subject_id):
    path = os.path.join(DATA_DIR, f"S{subject_id}", f"S{subject_id}.pkl")
    with open(path, "rb") as f:
        data = pickle.load(f, encoding="latin1")
    chest, wrist = data["signal"]["chest"], data["signal"]["wrist"]
    label = np.asarray(data["label"], dtype=int).ravel()

    # chest: filter whole signals
    ecg, _ = nk.ecg_process(np.asarray(chest["ECG"], float).ravel(), sampling_rate=FS_CHEST)
    eda_c, _ = nk.eda_process(np.asarray(chest["EDA"], float).ravel(), sampling_rate=FS_CHEST)
    rsp, _ = nk.rsp_process(np.asarray(chest["Resp"], float).ravel(), sampling_rate=FS_CHEST)
    try:
        emg, _ = nk.emg_process(np.asarray(chest["EMG"], float).ravel(), sampling_rate=FS_CHEST)
        emg_clean, emg_onsets = emg["EMG_Clean"].to_numpy(), emg["EMG_Onsets"].to_numpy()
    except Exception:
        emg_clean = emg_onsets = None
    temp_c = clean_temperature(np.asarray(chest["Temp"], float).ravel())

    # wrist: filter whole signals
    ppg, _ = nk.ppg_process(np.asarray(wrist["BVP"], float).ravel(), sampling_rate=FS_BVP)
    ppg_peaks = ppg["PPG_Peaks"].to_numpy()
    eda_w_raw = np.asarray(wrist["EDA"], float).ravel()
    temp_w_raw = np.asarray(wrist["TEMP"], float).ravel()
    try:
        eda_w, _ = nk.eda_process(eda_w_raw, sampling_rate=FS_EDA_W)
        tonic_w = eda_w["EDA_Tonic"].to_numpy()
    except Exception:
        tonic_w = None

    rows = []
    for lab, seg_start, seg_end in find_segments(label):
        start = seg_start
        while start + WLEN <= seg_end:
            c = slice(start, start + WLEN)                                   # chest 700 Hz
            # Map the window's start/length from the 700 Hz chest timeline to each wrist signal's
            # own sampling rate, so all signals stay aligned to the same 60 s time span
            b0 = int(round(start * FS_BVP / FS_CHEST)); b1 = min(b0 + WINDOW_SEC * FS_BVP, len(ppg_peaks))
            e0 = int(round(start * FS_EDA_W / FS_CHEST)); e1 = min(e0 + WINDOW_SEC * FS_EDA_W, len(eda_w_raw))
            f = {}

            # chest ECG (HRV)
            f.update(tag("c", hrv_features(ecg["ECG_R_Peaks"].values[c], FS_CHEST)))

            # chest EDA
            clean = eda_c["EDA_Clean"].values[c]; tonic = eda_c["EDA_Tonic"].values[c]; phasic = eda_c["EDA_Phasic"].values[c]
            scr = np.flatnonzero(eda_c["SCR_Peaks"].values[c] == 1)
            eda_f = {"SCL_mean": np.mean(tonic), "SCL_std": np.std(tonic),
                     "EDA_mean": np.mean(clean), "EDA_std": np.std(clean),
                     "EDA_min": np.min(clean), "EDA_max": np.max(clean), "EDA_range": np.ptp(clean),
                     "EDA_slope": np.polyfit(np.arange(len(clean)), clean, 1)[0],
                     "SCR_count": int(len(scr)),
                     "SCR_amp_mean": float(np.mean(phasic[scr])) if len(scr) else 0.0,
                     "SCR_amp_std": float(np.std(phasic[scr])) if len(scr) > 1 else 0.0,
                     "SCR_rate": len(scr) / WINDOW_SEC * 60.0,
                     "EDA_psd_power": band_power(clean, FS_CHEST, 0.05, 0.5)[0]}
            if len(scr) >= 2:                                     # Setz-style inter-peak interval,
                ipi = np.diff(scr) / FS_CHEST                     # kept identical to the chest-only
                eda_f["SCR_ipi_mean"] = float(np.mean(ipi))       # notebook
            else:
                eda_f["SCR_ipi_mean"] = np.nan
            f.update(tag("c", eda_f))

            # chest EMG (few features, from the cleaned signal)
            if emg_clean is not None:
                seg = emg_clean[c]
                f.update(tag("c", {"EMG_mav": np.mean(np.abs(seg)),                 # mean absolute value
                                   "EMG_rms": np.sqrt(np.mean(seg ** 2)),           # root mean square
                                   "EMG_wl": np.sum(np.abs(np.diff(seg))),          # waveform length
                                   "EMG_activations": int(np.nansum(emg_onsets[c] == 1))}))
            else:
                f.update(tag("c", {"EMG_mav": np.nan, "EMG_rms": np.nan,
                                   "EMG_wl": np.nan, "EMG_activations": np.nan}))

            # chest RESP
            rr = rsp["RSP_Rate"].values[c]; phase = rsp["RSP_Phase"].values[c]
            insp, exp = np.count_nonzero(phase == 1), np.count_nonzero(phase == 0)
            power, peak = band_power(rsp["RSP_Clean"].values[c], FS_CHEST, 0.1, 0.5)
            f.update(tag("c", {"RESP_rate_mean": np.nanmean(rr), "RESP_rate_std": np.nanstd(rr),
                               "RESP_amp_mean": np.nanmean(rsp["RSP_Amplitude"].values[c]),
                               "RESP_IE_ratio": insp / exp if exp > 0 else np.nan,
                               "RESP_psd_power": power, "RESP_peak_freq": peak}))

            # chest TEMP
            f.update(tag("c", temp_features(temp_c[c])))

            # wrist BVP (HRV, screened: noisy sensor)
            f.update(tag("w", hrv_features(ppg_peaks[b0:b1], FS_BVP, screen=True)))

            # wrist EDA (level only, 4 Hz — too coarse to resolve phasic SCR peaks)
            we = eda_w_raw[e0:e1]
            wf = {"EDA_mean": np.mean(we), "EDA_std": np.std(we), "EDA_min": np.min(we),
                  "EDA_max": np.max(we), "EDA_range": np.ptp(we),
                  "EDA_slope": np.polyfit(np.arange(len(we)), we, 1)[0] if len(we) > 1 else 0.0}
            if tonic_w is not None:
                wf["SCL_mean"] = np.mean(tonic_w[e0:e1]); wf["SCL_std"] = np.std(tonic_w[e0:e1])
            else:
                wf["SCL_mean"] = wf["SCL_std"] = np.nan
            f.update(tag("w", wf))

            # wrist TEMP
            f.update(tag("w", temp_features(temp_w_raw[e0:e1])))

            f.update(label=lab, label_name=LABEL_MAP[lab], subject_id=subject_id)
            rows.append(f)
            start += SLEN
    return rows

## Building the feature table  (slow cell)

This filters five chest signals and two wrist signals per subject


In [ ]:
subjects_to_run = SUBJECTS          
rows = []
for sid in subjects_to_run:
    try:
        subject_rows = process_subject(sid)
        rows.extend(subject_rows)
        print(f"S{sid}: {len(subject_rows)} windows")
    except Exception as e:
        print(f"S{sid}: ERROR ({e})")

features = pd.DataFrame(rows)
META = ["label", "label_name", "subject_id"]
feature_cols = [c for c in features.columns if c not in META]
print(f"\n{features.shape[0]} windows, {len(feature_cols)} features "
      f"({sum(c.startswith('c_') for c in feature_cols)} chest + "
      f"{sum(c.startswith('w_') for c in feature_cols)} wrist)")

S2: 390 windows
S3: 398 windows
S4: 399 windows
S5: 409 windows
S6: 407 windows
S7: 406 windows
S8: 407 windows
S9: 406 windows
S10: 422 windows
S11: 412 windows
S13: 411 windows
S14: 412 windows
S15: 413 windows
S16: 410 windows
S17: 421 windows

6123 windows, 64 features (40 chest + 24 wrist)


## Baseline-relative features

Each feature is also expressed relative to the subject's own baseline mean, and both versions are
kept. This applies to both devices, so chest and wrist are treated identically.


In [ ]:
rel_parts = []
for _, group in features.groupby("subject_id", sort=False):
    # Mean of each feature over this subject's own baseline windows only
    baseline_mean = group.loc[group["label_name"] == "baseline", feature_cols].mean()
    # Mean-centered on the subject's baseline (not z-scored): baseline windows end up ~0,
    # stress/amusement windows show the deviation from that subject's own resting level
    rel_parts.append(group[feature_cols] - baseline_mean)

relative = pd.concat(rel_parts).sort_index()
relative.columns = [f"{c}_rel" for c in feature_cols]
features = pd.concat([features, relative], axis=1)
print(f"{features.shape[0]} windows, {features.shape[1]} columns "
      f"({len(feature_cols)} absolute + {len(feature_cols)} relative + {len(META)} meta)")

6123 windows, 131 columns (64 absolute + 64 relative + 3 meta)


## Quality check

In addition to the missingness check, `c_RESP_rate_mean` is screened against a plausible
physiological range (see `RESP_RATE_MIN`/`RESP_RATE_MAX` above). Unlike EDA/ECG, `neurokit2`'s
respiration pipeline has no built-in artefact rejection, so a bad peak/trough detection in
`nk.rsp_process` can silently produce an implausible rate (e.g. close to 0 breaths/min). This
check does not drop or alter any window — it only flags them for inspection, since a handful of
outlier windows do not materially affect downstream modelling.


In [ ]:
print(features["label_name"].value_counts(), "\n")
missing = features.isna().sum()
print("Columns with missing values:")
print(missing[missing > 0] if missing.any() else "none")

# Flag-only check: implausible c_RESP_rate_mean windows are reported, not dropped or corrected
implausible_resp = (features["c_RESP_rate_mean"] < RESP_RATE_MIN) | (features["c_RESP_rate_mean"] > RESP_RATE_MAX)
n_bad = int(implausible_resp.sum())
print(f"\nWindows with implausible c_RESP_rate_mean (outside {RESP_RATE_MIN:.0f}-{RESP_RATE_MAX:.0f} "
      f"breaths/min): {n_bad} ({n_bad / len(features) * 100:.1f}%)")
if n_bad:
    print(features.loc[implausible_resp, ["subject_id", "label_name", "c_RESP_rate_mean"]]
          .sort_values("c_RESP_rate_mean").to_string(index=False))

label_name
baseline     3354
stress       1826
amusement     943
Name: count, dtype: int64 

Columns with missing values:
c_HRV_SampEn             48
c_SCR_ipi_mean         4653
c_RESP_IE_ratio           5
w_HR_mean              1945
w_HR_std               1945
w_HRV_MeanNN           1945
w_HRV_SDNN             1945
w_HRV_RMSSD            1945
w_HRV_pNN50            1945
w_HRV_SD1              1945
w_HRV_SD2              1945
w_HRV_SampEn           2735
w_HRV_LFHF             2439
c_HRV_SampEn_rel         48
c_SCR_ipi_mean_rel     5415
c_RESP_IE_ratio_rel       5
w_HR_mean_rel          1945
w_HR_std_rel           1945
w_HRV_MeanNN_rel       1945
w_HRV_SDNN_rel         1945
w_HRV_RMSSD_rel        1945
w_HRV_pNN50_rel        1945
w_HRV_SD1_rel          1945
w_HRV_SD2_rel          1945
w_HRV_SampEn_rel       2735
w_HRV_LFHF_rel         2439
dtype: int64

Windows with implausible c_RESP_rate_mean (outside 4-40 breaths/min): 42 (0.7%)
 subject_id label_name  c_RESP_rate_mean
          8   b

## Saving

In [12]:
features.to_csv(os.path.join(OUT_DIR, "features_all.csv"), index=False)
print(f"Saved {features.shape[0]} rows to {os.path.join(OUT_DIR, 'features_all.csv')}")

Saved 6123 rows to combined_outputs\features_all.csv
